In [1]:
# manual_code_mapping.py
"""
Manually map common conditions to ICD-10 codes.
"""

import json

# Manual mapping of conditions to ICD-10 codes
MANUAL_MAPPING = {
    # Cardiovascular
    'heart failure': 'I509',
    'congestive heart failure': 'I509',
    'myocardial infarction': 'I219',
    'heart attack': 'I219',
    'atrial fibrillation': 'I480',
    'hypertension': 'I10',
    'high blood pressure': 'I10',
    'hyperlipidemia': 'E785',
    'coronary artery disease': 'I2510',
    'angina': 'I200',
    
    # Metabolic
    'diabetes': 'E119',
    'diabetes mellitus': 'E119',
    'type 2 diabetes': 'E119',
    'type 1 diabetes': 'E101',
    'obesity': 'E669',
    
    # Respiratory
    'pneumonia': 'J189',
    'copd': 'J449',
    'chronic obstructive pulmonary disease': 'J449',
    'asthma': 'J45909',
    
    # Renal
    'chronic kidney disease': 'N189',
    'ckd': 'N189',
    'acute kidney injury': 'N179',
    'renal failure': 'N189',
    
    # Other
    'sepsis': 'A419',
    'stroke': 'I639',
    'cancer': 'C809',
    'anemia': 'D649',
    'pain': 'R529',
}

def apply_manual_mapping(trials):
    """Apply manual code mapping to trials."""
    fixed = []
    total_fixed = 0
    
    for trial in trials:
        fixed_criteria = []
        for c in trial.get('criteria', []):
            raw_text = c.get('raw_entity', '').lower()
            
            # Check if any mapping matches
            matched = False
            for condition, code in MANUAL_MAPPING.items():
                if condition in raw_text:
                    c['entity_code'] = code
                    c['entity_type'] = 'diagnosis'
                    matched = True
                    total_fixed += 1
                    break
            
            fixed_criteria.append(c)
        
        fixed.append({
            'nct_id': trial.get('nct_id'),
            'title': trial.get('title', ''),
            'conditions': trial.get('conditions', []),
            'phase': trial.get('phase', 'NA'),
            'sample_size': trial.get('sample_size', 100),
            'criteria': fixed_criteria
        })
    
    print(f"✅ Fixed {total_fixed} criteria with manual mapping")
    return fixed

def main():
    import os
    from config import Config
    cfg = Config()
    
    # Load trials
    train_path = f"{cfg.TRIALS_DATA_DIR}/structured_clinical_trials.json"
    eval_path = f"{cfg.TRIALS_DATA_DIR}/structured_clinical_trials_eval.json"
    
    with open(train_path, 'r') as f:
        train_trials = json.load(f)
    with open(eval_path, 'r') as f:
        eval_trials = json.load(f)
    
    print(f"Loaded {len(train_trials)} training trials, {len(eval_trials)} eval trials")
    
    # Apply manual mapping
    fixed_train = apply_manual_mapping(train_trials)
    fixed_eval = apply_manual_mapping(eval_trials)
    
    # Save
    with open(train_path, 'w') as f:
        json.dump(fixed_train, f, indent=2)
    with open(eval_path, 'w') as f:
        json.dump(fixed_eval, f, indent=2)
    
    print("✅ Saved fixed trials")

if __name__ == "__main__":
    main()

Loaded 149 training trials, 38 eval trials
✅ Fixed 243 criteria with manual mapping
✅ Fixed 58 criteria with manual mapping
✅ Saved fixed trials
